# Проект по дисциплине "Теория конечных графов" группа№3

Ладнюк Кира

Гареев Эльдар

Егоров Иван

# 0. Подготовка.

Импортируем необходимые библиотеки

In [148]:
import os
from collections import defaultdict, deque
import random
import numpy as np
from random import randint, random, sample, choice

Напишем функцию для обработки неориентированных графов. Графы будем хранить в виде словаря. При возникновении ошибки будет выходить сообщение об ошибке.

In [149]:
def make_graph(path):
    # Создаем словарь, который по умолчанию будет присваивать ключу пустое множество
    graph = defaultdict(set)
    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            for line in file:
                # Убираем лишние символы в начале и конце строки
                line = line.strip()

                # Пропускаем комментарии
                if line.startswith('#') or not line:
                    continue

                u, v = line.split()
                u = int(u)
                v = int(v)

                # Считаем общее количество ребер (включая кратные)
                all_edges += 1

                # Если ребро (u, v) ещё не было добавлено в словарь, добавляем (u, v) и (v, u)
                if v not in graph[u]:

                    graph[u].add(v)
                    graph[v].add(u)

                    # Считаем уникальные ребра
                    unique_edges += 1


    except FileNotFoundError:
        print(f"ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"произошла ошибка при обработке файла: {e}")
        return None

    return  dict(graph), unique_edges,  all_edges

Напишем функцию для генерации случайных графов. В качестве аргументов будем передавать желаемое количество графов:

In [150]:
def generate_graphs(n=0):
    if not n:
        n = randint(5, 15)
    graphs = []

    for _ in range(n):
        # Выбираем число вершин, равномерно распределенное от 20 до 50
        v = randint(20, 50)
        vertices = [i for i in range(v)]
        # Выбираем количество ребер как максимально возможное, умноженное на коэффициент от 0 до 1
        e = int(((v * (v - 1)) / 2) * random())
        graph = {vertice: set() for vertice in vertices}
        while e:
            u, v = sample(vertices, 2)
            if v not in graph[u]:
                graph[u].add(v)
                graph[v].add(u)
                e -= 1
        graphs.append(graph)
    return graphs

Будем использовать файл WikiVote в качестве примера:

In [151]:
file = "./Wiki-Vote.txt"
graph, edges, _ = make_graph(file)

# Анализ структуры сети
## Задание А
### _Часть 1_

Для каждой из сетей определить следующие характеристики:
Число вершин, число рёбер, плотность (отношение числа рёбер к максимально возможному числу рёбер), число компонент слабой связности, долю вершин в максимальной по мощности компоненте слабой связности. Для ориентированных графов определить число компонент сильной связности и долю вершин графа в наибольшей компоненте сильной связности компоненте.

Начнем с числа вершин и ребер:
  - кол-во вершин = кол-во ключей в словаре
  - кол-во ребер =  каждое ребро (A, B) хранится дважды, тогда общее количество рёбер можно получить как сумма мощностей множеств, делённая на 2
  - плотность = отношение числа рёбер к максимально возможному числу рёбер

In [152]:
num_vertex = len(graph)
max_num_vertex = num_vertex * (num_vertex - 1) // 2
density = edges / max_num_vertex if max_num_vertex > 0 else 0.0

print(f"Кол-во вершин: {num_vertex}, кол-во ребер: {edges}, плотность: {density}")

Кол-во вершин: 7115, кол-во ребер: 100762, плотность: 0.003981420144693063


todo: сделать визуализацию


Поиск компонент слабой связности. Для этого:
1. Реализуем DFS с помощью стека
2. Запускаем DFS по всем ключам словаря.
3. Функция вернет списков из списков, где каждый элемент - комппонента слабой связности.

In [153]:
def dfs_stack(graph):

    visited = set()
    components = []

    for u in graph:

        if u not in visited:
            stack = [u]
            component = []

            while stack:
                cur = stack.pop()

                if cur not in visited:
                    visited.add(cur)
                    component.append(cur)

                    for v in graph[cur]:
                        if v not in visited:
                            stack.append(v)

            components.append(component)
    return components


Рассмотрим сколько компонент слабой связности у нас получилось: для этого выведем длину получишегося массива.

In [154]:
components = dfs_stack(graph)
print(f'Число компонент слабой связности в графе равно {len(components)}')

Число компонент слабой связности в графе равно 24


Найдем долю вершин в максимальной по мощности компоненте слабой связности:
1. Найдем компоненту с максимальным числом вершин
2. Поделим мощность компоненты с максимальным числом вершин на общее количество вершин

In [155]:
max_size = max(len(c) for c in components)
fraction = max_size / num_vertex
print(f'Доля вершин в компоненте слабой связности равна {(fraction * 100):.2f}%')

Доля вершин в компоненте слабой связности равна 99.31%


### _Часть 2, пункт а_

Для наибольшей компоненты слабой связности оценить значения диаметра сети, 90 процентиля расстояния (геодезического) между вершинами графа. Оценку провести на основании **двойного прохода BFS** (the double sweep): для случайно выбранного узла найти максимально удаленный узел _a_, а затем найти узел _b_, максимально удаленный от _a_. За диаметр принять эксцентриситет вершины _ecc(a) = d(b,a)_

Для решения этого пункта:

1. Найдем наибольшую компоненту слабой связности
2. Используем алгоритм BFS
3. Выберем случайную вершину u, найдем самую удаленную от нее вершину v (первый проход BFS).
4. Найдем самую удаленную от v вершину w (второй проход BFS)

Тогда расстояние d(v, w) будет приближенным диаметром

Функция для нахождения самой большой компоненты слабой связности. На выход возвращает подграф в виде словаря.

In [156]:
def subgraph(components, graph):
    comp = max(components, key=len)
    nodes = set(comp)
    ans = {}

    # Переносим в ответ данные о тех ребрах, которые соединяют вершины наибольшей компоненты слабой связности
    for node in comp:
        ans[node] = {i for i in graph.get(node, set()) if i in nodes}

    return ans

In [157]:
large_comp = subgraph(components, graph)
print(f'Размер самой большой компоненты слабой связности равен {len(large_comp)}')

Размер самой большой компоненты слабой связности равен 7066


Реализация алгоритма BFS на очереди. Функция возвращает самую дальнюю вершину от выбранной и расстояния до всех вершин компоненты

In [158]:
def bfs_far(graph, u):

    dist = {u: 0}
    queue = deque([u])
    u = u
    max_dist = 0

    while queue:

        cur = queue.popleft()

        for v in graph.get(cur, set()):
            if v not in dist:

                dist[v] = dist[cur] + 1
                queue.append(v)
                if dist[v] > max_dist:
                    u = v
                    max_dist = dist[v]

    return u, dist


Выбираем две случайные вершины и прогоняем два раза bfs на них

In [159]:
def dbl_swp_diam(graph):
    if not graph:
        return 0

    u = choice(list(graph))
    v, _ = bfs_far(graph, u)
    _, ans = bfs_far(graph, v)

    return max(ans.values())

In [160]:
diameter = dbl_swp_diam(large_comp)
print(f'Диаметр сети, найденный с помощью двух вершин, равен {diameter}')

Диаметр сети, найденный с помощью двух вершин, равен 7


### _Часть 2, пункт b_
Вычисление 90-процентиля расстояний между случайными вершинами:
  1. Выбираем 1000 или 500 (по умолчанию) случайных пар вершин
  2. Для каждой пары делаем BFS, реализованный раннее
  3. Сортируем расстояния и находим 90-процентиль.

In [161]:
def percentile90(graph, n=500):

    nodes = list(graph.keys()) if hasattr(graph, 'keys') else list(graph)

    distances = []

    for _ in range(n):
        # Выберем две случайные вершины
        u, v = sample(nodes, 2)

        # Посчитаем расстояние с помощью BFS
        _, d = bfs_far(graph, u)

        if v in d:
            distances.append(d[v])

    # Используем функцию из библиотеки numpy, чтобы найти n-процентиль выборки
    return int(np.percentile(distances, 90)) if distances else 0

In [162]:
ans = percentile90(large_comp)

print(f'90-Процентиль расстояний в большой компоненте равен {ans}')

90-Процентиль расстояний в большой компоненте равен 4


### _Часть 2, пункт c_

Используем Snowball Sampling: будем добавлять к выборке первые n (500) вершин, которые посетим из некоторых начальных. Сформируем из выбранных вершин и ребер, соединяющих их, компоненту, а затем получим 90-процентиль расстояний в этой компоненте.

In [163]:
np.random.seed(26)
def snowball(graph, n=500):
    if not graph: return {}

    # Выбираем случаные вершины, с которых начнем
    s = sample(list(graph), min(3, len(graph)))
    visited, queue = set(s), deque(s)

    # Будем продолжать, пока не наберем n соседей или пока не иссякнет очередь
    while queue and len(visited) < n:
        c = queue.popleft()
        for i in graph.get(c, []):
            if i not in visited and len(visited) < n:
                visited.add(i)
                queue.append(i)

    sg = {node: set() for node in visited}
    for node in visited:
        sg[node] = {nb for nb in graph.get(node, []) if nb in visited}

    return sg

def snowball_analysis(graph, n=500):
    subgraph = snowball(graph, n)
    diameter = dbl_swp_diam(subgraph)
    percentile = percentile90(subgraph, n=100)
    return diameter, percentile

In [164]:
sn_dim, sn_per = snowball_analysis(large_comp)
print(f'90-Процентиль расстояний в снежном коме равен {sn_per}, диаметр равен {sn_dim}')

90-Процентиль расстояний в снежном коме равен 3, диаметр равен 3


### _Часть 3_
Посчитаем количество треугольников

In [ ]:
def triangles(graph):

    triangles = 0

    for u in graph:
        current = graph[u]
        for v in current:
            # Анализируем соседей
            if v > u:
                for w in current & graph[v]:
                    if w > v:
                        triangles += 1

    return triangles

In [166]:
tri_number = triangles(graph)
print(f'Количество полных подграфов на 3 вершинах равно {tri_number}')

Количество полных подграфов на 3 вершинах равно 608389


### _Часть 4_


Вычислим средний кластерный коэффициент с помощью формулы, указанной в задании

In [ ]:
def node_clust(graph, n):
    nodes = graph.get(n, set())
    neighbours = len(nodes)
    if neighbours < 2: return 0.0

    max = neighbours * (neighbours - 1) / 2
    unique = 0

    for u in nodes:
        for v in nodes:
            if u > v and v in graph.get(u, set()):
                unique += 1

    return unique / max

def avg_clust(graph):

    ans = 0.0
    count = 0

    for node in graph:
        coeff = node_clust(graph, node)
        ans += coeff
        count += 1

    return ans / count

In [ ]:
print(f'Средний кластерный коэффициент в компоненте составляет: {avg_clust(large_comp):4f}')
print(f'Средний кластерный коэффициент в графе составляет: {avg_clust(graph):4f}')

Средний кластерный коэффициент в компоненте составляет: 0.141875
Средний кластерный коэффициент в графе составляет: 0.140898


### _Часть 5_
Найдем минимальную, максимальную и среднюю степени вершин в графе

In [167]:
def calculate_node_degrees(graph):
    degrees = [len(u) for u in graph.values()]

    return min(degrees),max(degrees), sum(degrees) / len(degrees)

In [168]:
min_deg, max_deg, avg_deg = calculate_node_degrees(graph)
print(f'Минимальная степень вершины в графе равна {min_deg}')
print(f'Максимальная степень вершины в графе равна {max_deg}')
print(f'Средняя степень вершины в графе равна {avg_deg:.4f}')

Минимальная степень вершины в графе равна 1
Максимальная степень вершины в графе равна 1065
Средняя степень вершины в графе равна 28.3238


In [171]:
def gcc(g):
    total, count = 0, 0

    for u in g:
        n = g[u]
        k = len(n)

        if k < 2: 
            continue
        total += k * (k - 1) / 2

        for v in n:
            for w in n:
                if v > w and w in g.get(v,set()):
                    count += 1

    return count / total if total else 0.0

In [172]:
print(f'Глобальный кластерный коэффициент в графе составляет: {avg_clust(graph):4f}')

Глобальный кластерный коэффициент в графе составляет: 0.140898


In [173]:
def avg_clust(g, comp):
    if not comp: return 0.0

    total = 0.0
    nodes = set(comp)

    for u in comp:
        nb = [v for v in g.get(u, set()) if v in nodes]
        k = len(nb)
        if k < 2: continue

        edges = 0
        for i in range(k):
            for j in range(i+1, k):
                if nb[j] in g.get(nb[i], set()):
                    edges += 1

        total += (2 * edges) / (k * (k - 1))

    return total / len(comp)

In [174]:
def process_directed_graph_file(path):
    graph = defaultdict(set)

    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            for line in file:
                line = line.strip()

                if line.startswith('#') or not line:
                    continue

                u, v = line.split()
                u = int(u)
                v = int(v)


                all_edges += 1

                if u not in graph[v]:
                    graph[u].add(v)
                    unique_edges += 1

    except FileNotFoundError:
        print(f"ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"произошла ошибка при обработке файла: {e}")
        return None

    return dict(graph), unique_edges, all_edges

In [175]:
file = "./Wiki-Vote.txt"
graph, edges, _ = make_graph(file)

In [176]:
print(edges)

100762


In [177]:
def count_scc(graph):
    # Первый проход DFS для определения порядка завершения
    visited = set()
    order = []

    for u in graph:
        if u not in visited:
            stack = [(u, False)]

            while stack:
                cur, processed = stack.pop()
                if processed:
                    order.append(cur)
                    continue
                if cur in visited:
                    continue

                if cur not in visited:
                    visited.add(cur)
                    stack.append((cur, True))
                    for v in graph.get(cur, set()):
                        if v not in visited:
                            stack.append((v, False))

    # Инвертируем граф
    reversed_graph = defaultdict(set)
    for src in graph:
        for dst in graph[src]:
            reversed_graph[dst].add(src)

    # Второй проход DFS в обратном порядке по инвертированному графу
    visited = set()
    components = []

    for u in reversed(order):
        if u not in visited:
            stack = [u]
            visited.add(u)
            component = []

            while stack:
                cur = stack.pop()
                component.append(cur)

                for v in reversed_graph.get(cur, set()):
                    if v not in visited:
                        stack.append(v)
                        visited.add(v)

            components.append(component)
    return components

In [178]:
scc = count_scc(graph)

print(len(scc))

24
